# Training Notebook
This notebook was updated by the assistant.
- Add your training code below.

In [ ]:
# Minimal in-notebook training snippet using project modules
import sys, os
from pathlib import Path
# Ensure project root is on sys.path
sys.path.insert(0, str(Path('..').resolve()))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from model.lstm_model import SimpleLSTM
from core.utils import clean_text, create_vocab, text_to_sequences

data_path = Path('data/corpus.txt')
if not data_path.exists():
    raise FileNotFoundError(f'Data file not found: {data_path}')

text = clean_text(data_path.read_text(encoding='utf-8'))
word_to_idx, idx_to_word = create_vocab(text)
vocab_size = len(word_to_idx)
seq_length = 5
sequences = text_to_sequences(text, word_to_idx, seq_length)
sequences = np.array(sequences)
if len(sequences) == 0:
    raise ValueError('No sequences generated from text — check corpus and seq_length')

# Use a subset to keep this quick in the notebook
max_samples = 1000
sequences = sequences[:max_samples]
x = torch.tensor(sequences[:, :-1], dtype=torch.long)
y = torch.tensor(sequences[:, -1], dtype=torch.long)

model = SimpleLSTM(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 3
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output, _ = model(x)
    last_output = output[:, -1, :]
    loss = criterion(last_output, y)
    loss.backward()
    optimizer.step()
    print(f'epoch {epoch+1}/{epochs}, loss={loss.item():.4f}')

save_dir = Path('models')
save_dir.mkdir(exist_ok=True)
save_path = save_dir / 'notebook_lstm_weights.pth'
torch.save(model.state_dict(), str(save_path))
print(f'Saved model to {save_path}')

: 